# 10. Typ I- och typ II-fel

Ett test kan ge fel svar på två sätt: det kan larma när inget finns, och det kan missa något som finns.
I de två förra notebookarna såg du exempel på båda. Nu undersöker vi felen systematiskt, och svarar på den praktiska
frågan *"hur mycket data behöver jag?"*

**Efter den här notebooken kan du**
- förklara vad typ I-fel och typ II-fel är
- förklara vad ett tests styrka är och vad som påverkar den
- beskriva avvägningen mellan de två felen
- uppskatta genom simulering hur stort stickprov som behövs

**Förkunskaper:** [8. Hypotesprövning och p-värden](../08_hypotesprovning/08_hypotesprovning.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

## 10.1 Fyra möjliga utfall

Verkligheten är antingen att $H_0$ är sann eller att den är falsk. Testet antingen förkastar $H_0$ eller gör det inte.

|  | $H_0$ är sann (ingen effekt finns) | $H_0$ är falsk (en effekt finns) |
|---|---|---|
| **Testet förkastar $H_0$** | ❌ **Typ I-fel** (falskt larm), sannolikhet α | ✅ Rätt. Sannolikheten kallas **styrka** |
| **Testet förkastar inte $H_0$** | ✅ Rätt | ❌ **Typ II-fel** (missad effekt), sannolikhet β |

**Styrka** = 1 − β = sannolikheten att upptäcka en effekt som faktiskt finns.

Jämför med ett brandlarm: typ I-fel är att larmet går fast det inte brinner. Typ II-fel är att det brinner utan att larmet går.
Eller med rättegången från notebook 8: typ I är att fälla en oskyldig, typ II att fria en skyldig.

I verkligheten vet vi aldrig vilken kolumn vi befinner oss i. Men i en simulering bestämmer vi det själva.

## 10.2 Typ I-fel: när ingenting finns att hitta

Vi simulerar 2 000 experiment där två grupper jämförs med t-test. Båda grupperna dras ur **exakt samma fördelning**,
så $H_0$ är sann i vartenda experiment. Varje signifikant resultat är ett falskt larm.

🔮 **Gissa först:** hur ser histogrammet över de 2 000 p-värdena ut? En topp nära 1? Nära 0,5? Något annat?

In [ ]:
num_trials = 2000
n_per_group = 30        # ← ändra: pröva 10 och 300

group_a = rng.normal(0, 1, (num_trials, n_per_group))      # varje rad är ett experiment
group_b = rng.normal(0, 1, (num_trials, n_per_group))      # samma fördelning: H0 är sann
p_values = stats.ttest_ind(group_a, group_b, axis=1, equal_var=False).pvalue

plt.hist(p_values, bins=np.arange(0, 1.05, 0.05), edgecolor="white")
plt.axvline(0.05, color="red", linestyle="--", label="α = 0,05")
plt.xlabel("p-värde")
plt.ylabel("Antal experiment")
plt.title("p-värden när H0 är sann")
plt.legend()
plt.show()

print(f"Andel med p < 0,05: {(p_values < 0.05).mean():.1%}")

När $H_0$ är sann är p-värdena **jämnt fördelade** mellan 0 och 1. Alla värden är lika troliga, och därför hamnar 5 % av
dem under 0,05.

> **Risken för typ I-fel är α.** Den bestämmer du själv genom valet av signifikansnivå, och den påverkas inte av
> stickprovets storlek. (Kontrollera genom att ändra `n_per_group`.)

Ungefär vart tjugonde test av en nollhypotes som är sann ger alltså ett "signifikant" resultat. Det får stora följder när
man gör många tester, vilket vi återkommer till i [notebook 15](../15_kritisk_vardering/15_kritisk_vardering.ipynb).

✏️ **Din tur:** Hur stor andel av p-värdena ovan är under 0,01? Under 0,10? Stämmer det med påståendet att risken för typ I-fel är α?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for a in [0.01, 0.05, 0.10]:
    print(f"α = {a:.2f}: andel falska larm {(p_values < a).mean():.1%}")
```

</details>

## 10.3 Typ II-fel: när något finns men inte hittas

Nu lägger vi in en verklig skillnad. Värdena har standardavvikelsen 1, så en skillnad på 0,5 betyder
**en halv standardavvikelse**, vilket brukar räknas som en medelstor effekt.

Nu är $H_0$ falsk i varje experiment. Varje *icke*-signifikant resultat är en missad effekt.

In [ ]:
true_difference = 0.5      # ← ändra: pröva 0.2 och 0.8
n_per_group = 30           # ← ändra: pröva 10 och 100

group_a = rng.normal(0, 1, (num_trials, n_per_group))
group_b = rng.normal(true_difference, 1, (num_trials, n_per_group))
p_values_effect = stats.ttest_ind(group_a, group_b, axis=1, equal_var=False).pvalue

plt.hist(p_values_effect, bins=np.arange(0, 1.05, 0.05), edgecolor="white")
plt.axvline(0.05, color="red", linestyle="--", label="α = 0,05")
plt.xlabel("p-värde")
plt.ylabel("Antal experiment")
plt.title(f"p-värden när den sanna skillnaden är {true_difference} sd, n = {n_per_group} per grupp")
plt.legend()
plt.show()

power = (p_values_effect < 0.05).mean()
print(f"Styrka (effekten upptäcks): {power:.0%}")
print(f"Typ II-fel (effekten missas): {1 - power:.0%}")

Nu trycks p-värdena ihop mot noll, men långt ifrån alla hamnar under 0,05. Med en medelstor effekt och 30 personer per
grupp **missar testet effekten ungefär varannan gång**. Kom ihåg det nästa gång du läser att en studie "inte fann någon effekt".

🧪 **Labba:** Ändra en sak i taget.
1. Vad händer med styrkan när effekten är större (0.8)? Mindre (0.2)?
2. Vad händer när grupperna är större (100)? Mindre (10)?

## 10.4 Vad styr styrkan?

För att experimentera snabbare lägger vi simuleringen i en funktion.

In [ ]:
def share_rejected(true_difference, n_per_group, alpha=0.05, num_trials=2000):
    """Andel av num_trials simulerade experiment där H0 förkastas."""
    a = rng.normal(0, 1, (num_trials, n_per_group))
    b = rng.normal(true_difference, 1, (num_trials, n_per_group))
    p_values = stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue
    return (p_values < alpha).mean()

print(f"Ingen skillnad, n = 30:  H0 förkastas i {share_rejected(0, 30):.1%} av fallen  (typ I-fel)")
print(f"Skillnad 0,5, n = 30:    H0 förkastas i {share_rejected(0.5, 30):.1%} av fallen  (styrka)")

Styrkan som funktion av stickprovsstorleken, för tre olika effektstorlekar:

In [ ]:
sizes = [5, 10, 20, 30, 50, 75, 100, 150, 200]

for true_difference in [0.2, 0.5, 0.8]:
    powers = [share_rejected(true_difference, n) for n in sizes]
    plt.plot(sizes, powers, marker="o", label=f"sann skillnad {true_difference} sd")
plt.axhline(0.8, color="gray", linestyle="--", label="80 % styrka")
plt.xlabel("Antal per grupp")
plt.ylabel("Styrka")
plt.ylim(0, 1.02)
plt.legend()
plt.show()

- **Större stickprov ger högre styrka.**
- **Större effekter är lättare att upptäcka.** En stor effekt (0,8) syns redan med ett trettiotal per grupp.
  En liten effekt (0,2) upptäcks bara ungefär varannan gång även med 200 per grupp.

En vanlig tumregel är att planera för **80 % styrka**.

✏️ **Din tur:** Pröva dig fram med `share_rejected`: ungefär hur många per grupp behövs för 80 % styrka när den sanna
skillnaden är 0,5? Och när den är 0,2?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for n in [50, 60, 64, 70, 80]:
    print(f"skillnad 0,5, n = {n}: styrka {share_rejected(0.5, n):.0%}")
print()
for n in [200, 300, 400, 500]:
    print(f"skillnad 0,2, n = {n}: styrka {share_rejected(0.2, n):.0%}")
# Ungefär 64 per grupp för skillnaden 0,5 och ungefär 400 per grupp för 0,2.
# En effekt som är 2,5 gånger mindre kräver drygt 6 gånger så mycket data.
```

</details>

## 10.5 Avvägningen mellan felen

Om falska larm är så illa, varför inte sätta α = 0,001?

🔮 **Gissa först:** vad händer med styrkan om vi sänker α?

In [ ]:
print("Sann skillnad 0,5 sd, n = 30 per grupp")
for alpha in [0.10, 0.05, 0.01, 0.001]:
    power = share_rejected(0.5, 30, alpha=alpha)
    print(f"α = {alpha:<6}  typ I-fel {alpha:>6.1%}   styrka {power:>4.0%}   typ II-fel {1 - power:>4.0%}")

Strängare krav för att larma ger färre falska larm, men också fler missade effekter. **Med en given mängd data kan man
inte minska båda felen samtidigt.** Det enda sättet att förbättra båda är att samla in mer data.

Vilket fel som är värst beror på situationen.

✏️ **Din tur:** Ingen kod. Vilket fel är allvarligast i varje fall, och vad säger det om valet av α?

**A.** Ett test av om ett nytt läkemedel har allvarliga biverkningar ($H_0$: inga biverkningar).

**B.** En domstol som prövar om den åtalade är skyldig ($H_0$: oskyldig).

**C.** En webbutik som testar om en ny knappfärg ger fler klick ($H_0$: ingen skillnad). Att byta färg kostar nästan ingenting.

<details>
<summary>💡 <b>Svar</b></summary>

**A.** Typ II-felet (att missa biverkningar som finns) är allvarligast. Här kan man acceptera ett högre α för att få bättre styrka.

**B.** Typ I-felet (att fälla en oskyldig) anses allvarligast, därav kravet "ställt utom rimligt tvivel", vilket motsvarar ett mycket lågt α.

**C.** Inget av felen är särskilt dyrt. Ett falskt larm innebär bara ett färgbyte som inte gjorde någon skillnad.
Ett högre α kan vara rimligt.

</details>

## 10.6 Vad betyder detta när du läser en studie?

En sista simulering. Anta att en verklig men måttlig effekt (0,3 sd) undersöks i en liten studie med 20 per grupp.
Vi tittar bara på de studier som blev signifikanta – det är ju de som oftast publiceras.

In [ ]:
true_difference, n_per_group = 0.3, 20

a = rng.normal(0, 1, (5000, n_per_group))
b = rng.normal(true_difference, 1, (5000, n_per_group))
p_vals = stats.ttest_ind(a, b, axis=1, equal_var=False).pvalue
observed_diff = b.mean(axis=1) - a.mean(axis=1)
significant = p_vals < 0.05

print(f"Styrka: {significant.mean():.0%}")
print(f"Sann skillnad:                                   {true_difference}")
print(f"Genomsnittlig skattad skillnad, alla studier:    {observed_diff.mean():.2f}")
print(f"Genomsnittlig skattad skillnad, bara signifikanta: {observed_diff[significant].mean():.2f}")

Bland alla studier är skattningen rätt i genomsnitt. Men de som blev signifikanta **överskattar effekten kraftigt**.
Förklaringen är att en liten studie bara kan bli signifikant om den av en slump råkar få en ovanligt stor skillnad.

Små studier med signifikanta resultat ska alltså läsas med försiktighet: både för att de kan vara falska larm, och för att
effekten troligen är överdriven även om den finns.

## Sammanfattning

- **Typ I-fel:** förkasta en sann $H_0$ (falskt larm). Sannolikheten är α, som du väljer själv.
- **Typ II-fel:** inte förkasta en falsk $H_0$ (missad effekt). Sannolikheten är β.
- **Styrka** = 1 − β. Den ökar med större stickprov, större effekt och högre α.
- Lägre α ger färre typ I-fel men fler typ II-fel. Bara mer data förbättrar båda.
- Ett icke-signifikant resultat från en liten studie säger nästan ingenting.

### Kontrollfrågor
1. En forskare gör ett test med 25 % styrka och får p = 0,30. Vad kan hen säga?
2. Påverkas risken för typ I-fel av stickprovets storlek?
3. Varför räcker det inte att veta α för att bedöma hur pålitligt ett test är?

<details>
<summary>💡 <b>Svar</b></summary>

1. Nästan ingenting. Testet skulle ha missat en verklig effekt tre gånger av fyra, så ett icke-signifikant resultat är väntat
   oavsett om effekten finns eller inte.
2. Nej, den är α oavsett n. Det är styrkan som påverkas av n.
3. α säger bara hur ofta testet larmar falskt. Hur ofta det hittar verkliga effekter (styrkan) beror på stickprovsstorlek och effektstorlek.

</details>

**Nästa notebook:** [11. Korstabeller och chi-två-test](../11_korstabeller_chi2/11_korstabeller_chi2.ipynb)